In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TABLE_DECIMALS = 3
pd.set_option("display.float_format", lambda x: f"{x:.{TABLE_DECIMALS}f}")

ID_LIKE_COLS = {
    "run_id", "broad_no", "feature_id", "chat_id", "snapshot_id",
    "k", "cluster", "kmeans_cluster", "model_cluster", "session_type_cluster",
    "n", "n_sessions", "n_minutes", "n_rows", "result",
    "rule_score", "max_zero_run", "candidate_minutes", "candidate_intervals"
}

def _round_numeric_for_export(table, decimals=TABLE_DECIMALS):
    if isinstance(table, pd.Series):
        out = table.to_frame().copy()
    else:
        out = table.copy()

    for col in out.columns:
        if not pd.api.types.is_numeric_dtype(out[col]):
            continue

        # ID/count 성격의 column은 소수점이 붙지 않도록 정수형으로 유지
        if str(col) in ID_LIKE_COLS:
            s = out[col]
            if s.notna().all() and np.allclose(s.astype(float), np.round(s.astype(float))):
                out[col] = s.astype("Int64")
            else:
                out[col] = s.round(decimals)
        elif pd.api.types.is_float_dtype(out[col]):
            out[col] = out[col].round(decimals)

    return out

# 발표용 그래프/표 저장 폴더
FIG_SAVE_DIR = Path.cwd() / "eda_slide_plots"
TABLE_SAVE_DIR = Path.cwd() / "eda_slide_tables"
FIG_SAVE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_SAVE_DIR.mkdir(parents=True, exist_ok=True)

# 필요한 발표용 출력만 명시적으로 저장한다.
def save_fig(fig, filename):
    path = FIG_SAVE_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    print(f"saved figure: {path}")
    return path

def save_table(table, filename):
    path = TABLE_SAVE_DIR / filename
    table_to_save = _round_numeric_for_export(table)
    table_to_save.to_csv(
        path,
        index=False,
        encoding="utf-8-sig",
        float_format=f"%.{TABLE_DECIMALS}f"
    )
    print(f"saved table: {path}")
    return path

print(f"EDA slide plot folder: {FIG_SAVE_DIR}")
print(f"EDA slide table folder: {TABLE_SAVE_DIR}")

# 총 20개의 파일이 있고 파일의 특성상 
# 내부에 시간대는 연속되지만 session별로 streamer가 연속되지 않는다.
# 일단 전부 읽어오고 추후 처리

# 해당 파일명 형태와 같은 모든 것들 폴더에서 읽어옴
def file_reading(dir, file_name):
    DATA_DIR = Path(dir)
    files = sorted(DATA_DIR.glob(file_name))

    if len(files) == 0:
        raise FileNotFoundError("*번째의 데이터파일 없음")
    frames = []

    for f in files:
        tmp = pd.read_excel(f)
        # 파일명도 새 column으로 저장
        tmp["source_file"] = f.name

        # 파일 내부에 run_id가 없거나 전부 비어 있으면 파일명에서 추출
        if "run_id" not in tmp.columns or tmp["run_id"].isna().all():
            m = re.search(r"Run_(\d+)_Features", f.name)
            tmp["run_id"] = int(m.group(1)) if m else np.nan

        frames.append(tmp)

    raw = pd.concat(frames, ignore_index=True)
    meta_data = pd.DataFrame({"loaded_file_cnt":[len(files)], 
                   "row_cnt": [len(raw)],
                   "column_cnt" : [raw.shape[1]]})
    return raw , meta_data

raw, meta_data = file_reading("./datasets","Run_*_Features.xlsx")

# 파일 20개 읽음, row 20519개, column 18개
print(meta_data)
print(raw.head(3))